In [1]:
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "python-dotenv", "--break-system-packages", "-q"])
    from dotenv import load_dotenv
    load_dotenv()

In [2]:
# One-time installs (uncomment on first run):
# %pip install numpy==1.26.4 "sentinelhub[AWS]==3.8.3" eo-learn==1.3.1 geopandas==0.13.2 \
#     fiona==1.9.6 "pandas>=2.0,<2.2" opencv-python-headless "shapely>=2.0" "rasterio>=1.3" \
#     tqdm "omnicloudmask>=1.7" "setuptools<81"

import os, subprocess, shutil, glob

os.chdir(os.path.expanduser("~/projects/iride_onboard-burnscar-mapper"))

# --- fetch the official PhiSat-2 simulator code (AI4EO/orbitalAI) ---
if not os.path.exists("phisat2_utils.py"):
    if not os.path.isdir("orbitalAI"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/AI4EO/orbitalAI.git"], check=True)
    shutil.copy("orbitalAI/phisat-2/phisat2_constants.py", ".")
    shutil.copy("orbitalAI/phisat-2/phisat2_utils.py", ".")
    if os.path.isdir("orbitalAI/phisat-2/executables"):
        shutil.copytree("orbitalAI/phisat-2/executables", "executables", dirs_exist_ok=True)
        for f in glob.glob("executables/*"):
            os.chmod(f, 0o755)

import math, json, zipfile, logging, warnings
import datetime as dt
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds

from eolearn.core import EOPatch, EOTask, FeatureType, MapFeatureTask, RemoveFeatureTask
from eolearn.features.utils import spatially_resize_image as resize_images
from eolearn.io import SentinelHubInputTask
from sentinelhub import (BBox, DataCollection, SentinelHubCatalog, SHConfig,
                         get_utm_crs, wgs84_to_utm)

from phisat2_constants import (BBOX_SIZE, PHISAT2_RESOLUTION, S2_BANDS,
                               S2_PAN_BANDS, S2_RESOLUTION, ProcessingLevels)
from phisat2_utils import (AddMetadataTask, AddPANBandTask,
                           AlternativePhisatCalculationTask, BandMisalignmentTask,
                           CalculateRadianceTask, CalculateReflectanceTask,
                           CropTask, PhisatCalculationTask, SCLCloudTask)

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("phisat2_fire")
print("imports OK | cwd:", os.getcwd())

/root/projects/iride_onboard-burnscar-mapper/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/projects/iride_onboard-burnscar-mapper/.venv/lib/python3.12/site-packages/fs/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore


imports OK | cwd: /root/projects/iride_onboard-burnscar-mapper


/root/projects/iride_onboard-burnscar-mapper/.venv/lib/python3.12/site-packages/eolearn/io/sentinelhub_process.py:36: SHDeprecationWarning: The module `sentinelhub.type_utils` is deprecated, use `sentinelhub.types` instead.
  from sentinelhub.type_utils import RawTimeIntervalType
/root/projects/iride_onboard-burnscar-mapper/phisat2_constants.py:37: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  WORLD_GDF = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))


In [3]:
# ---------------------------------------------------------------- constants --
IDX_B03 = S2_BANDS.index("B03")   # green
IDX_B04 = S2_BANDS.index("B04")   # red
IDX_B08 = S2_BANDS.index("B08")   # NIR

MASK_CLEAR, MASK_BURNT, MASK_CLOUD, MASK_CLOUD_SHADOW, MASK_WATER, MASK_NODATA = 0, 2, 3, 4, 5, 255
MASK_COLORMAP = {0: (200,200,200,255), 2: (160,30,30,255), 3: (255,255,255,255),
                 4: (60,60,90,255), 5: (30,90,200,255), 255: (0,0,0,0)}

CDSE_BASE_URL = "https://sh.dataspace.copernicus.eu"
CDSE_TOKEN_URL = ("https://identity.dataspace.copernicus.eu/auth/realms/CDSE"
                  "/protocol/openid-connect/token")

S2L1C_COLLECTION = DataCollection.SENTINEL2_L1C.define_from(
    "SENTINEL2_L1C_CDSE", service_url=CDSE_BASE_URL)
S2L2A_COLLECTION = DataCollection.SENTINEL2_L2A.define_from(
    "SENTINEL2_L2A_CDSE", service_url=CDSE_BASE_URL)

EXPORT_BANDS = ["B02", "B03", "B04", "B05", "B06", "B07", "B08"]   # b,g,r,re1,re2,re3,nir
EXPORT_IDX = [S2_PAN_BANDS.index(b) for b in EXPORT_BANDS]
EXPORT_AS_UINT16 = True

# ------------------------------------------------------------ EFFIS loading --
COUNTRY_CANDIDATES = ["iso2", "ISO2", "COUNTRY", "country", "Country", "CNTR", "cntr_code"]
DATE_CANDIDATES = ["FIREDATE", "firedate", "FIRE_DATE", "initialdate", "INITIALDATE", "DATE"]
AREA_CANDIDATES = ["AREA_HA", "area_ha", "Area_HA", "AREA", "area"]
ID_CANDIDATES = ["id", "ID", "Id", "OBJECTID", "fid"]

def _find_col(gdf, candidates):
    lower = {c.lower(): c for c in gdf.columns}
    return next((lower[c.lower()] for c in candidates if c.lower() in lower), None)

def _read_vector(path):
    p = Path(path)
    if p.suffix.lower() == ".zip":
        with zipfile.ZipFile(p) as zf:
            members = zf.namelist()
        vector = next((m for m in members
                       if m.lower().endswith((".geojson", ".json", ".shp", ".gpkg"))), None)
        if vector is None:
            raise ValueError(f"No GeoJSON/shapefile inside {path}: {members}")
        return gpd.read_file(f"zip://{p}!{vector}")
    return gpd.read_file(path)

def load_effis(path, countries=None, start_date=None, end_date=None, min_area_ha=0.0):
    gdf = _read_vector(path)
    gdf = (gdf.set_crs(4326) if gdf.crs is None else gdf).to_crs(4326)
    c_country, c_date = _find_col(gdf, COUNTRY_CANDIDATES), _find_col(gdf, DATE_CANDIDATES)
    c_area, c_id = _find_col(gdf, AREA_CANDIDATES), _find_col(gdf, ID_CANDIDATES)
    if c_date is None:
        raise ValueError(f"No fire-date column found in {list(gdf.columns)}")
    out = gpd.GeoDataFrame(geometry=gdf.geometry, crs=gdf.crs)
    out["country"] = gdf[c_country].astype(str).str.strip().str.upper() if c_country else "NA"
    dates = pd.to_datetime(gdf[c_date], errors="coerce", format="mixed")
    if dates.isna().mean() > 0.5:
        dates = pd.to_datetime(gdf[c_date], errors="coerce", dayfirst=True)
    out["fire_date"] = dates.dt.tz_localize(None) if getattr(dates.dt, "tz", None) is not None else dates
    out["area_ha"] = pd.to_numeric(gdf[c_area], errors="coerce") if c_area else np.nan
    out["fire_id"] = (gdf[c_id].astype(str) if c_id
                      else pd.Series(range(len(gdf)), index=gdf.index).astype(str))
    out["fire_id"] = "effis_" + out["fire_id"].str.replace(r"[^\w\-]", "_", regex=True)
    out = out[out.geometry.notna() & out["fire_date"].notna()]
    if countries:  out = out[out["country"].isin({c.upper() for c in countries})]
    if start_date: out = out[out["fire_date"] >= pd.Timestamp(start_date)]
    if end_date:   out = out[out["fire_date"] <= pd.Timestamp(end_date)]
    if min_area_ha: out = out[out["area_ha"].fillna(0) >= min_area_ha]
    return out.sort_values("area_ha", ascending=False).reset_index(drop=True)

# ------------------------------------------- metadata without AWS S3 access --
SOLAR_IRRADIANCE = {"B02": 1959.66, "B03": 1823.24, "B04": 1512.06, "B05": 1424.64,
                    "B06": 1287.61, "B07": 1162.08, "B08": 1041.63}

def earth_sun_correction(date):
    doy = date.timetuple().tm_yday
    d = 1.0 - 0.01672 * math.cos(math.radians(0.9856 * (doy - 4)))
    return 1.0 / (d * d)

class AddMetadataFallbackTask(EOTask):
    def execute(self, eopatch):
        dim = len(eopatch.timestamp)
        eopatch.scalar["earth_sun_dist"] = np.array(
            [[earth_sun_correction(ts)] for ts in eopatch.timestamp], dtype=np.float64)
        for band in S2_BANDS:
            eopatch.scalar[f"sol_irr_{band}"] = np.full((dim, 1), SOLAR_IRRADIANCE[band])
        return eopatch

# ----------------------------------- OmniCloudMask on the native 10 m bands --
class OmniCloudMaskTask(EOTask):
    def __init__(self, inference_device=None, batch_size=1):
        self.inference_device, self.batch_size = inference_device, batch_size

    def execute(self, eopatch):
        from omnicloudmask import predict_from_array
        bands = eopatch[(FeatureType.DATA, "BANDS")]
        t, h, w, _ = bands.shape
        cloud = np.zeros((t, h, w, 1), np.uint8)
        shadow = np.zeros((t, h, w, 1), np.uint8)
        for ti in range(t):
            arr = np.stack([bands[ti, ..., IDX_B04], bands[ti, ..., IDX_B03],
                            bands[ti, ..., IDX_B08]], axis=0).astype(np.float32) * 10000.0
            kwargs = dict(batch_size=self.batch_size, apply_no_data_mask=True, no_data_value=0)
            if self.inference_device:
                kwargs["inference_device"] = self.inference_device
            pred = np.asarray(predict_from_array(arr, **kwargs))[0]
            cloud[ti, ..., 0] = ((pred == 1) | (pred == 2)).astype(np.uint8)
            shadow[ti, ..., 0] = (pred == 3).astype(np.uint8)
        eopatch[(FeatureType.MASK, "OCM_CLOUD")] = cloud
        eopatch[(FeatureType.MASK, "OCM_SHADOW")] = shadow
        return eopatch

# ------------------------------------------------------ water: SCL 6 | NDWI --
class WaterMaskTask(EOTask):
    def __init__(self, ndwi_threshold=0.05, use_scl_water=True):
        self.thr, self.use_scl = ndwi_threshold, use_scl_water

    def execute(self, eopatch):
        bands = eopatch[(FeatureType.DATA, "BANDS")]
        green, nir = bands[..., IDX_B03], bands[..., IDX_B08]
        denom = np.maximum(green + nir, 1e-6)
        water = ((green - nir) / denom) > self.thr
        if self.use_scl and "SCL" in eopatch.mask:
            water = water | (eopatch.mask["SCL"][..., 0] == 6)
        eopatch[(FeatureType.MASK, "WATER")] = water[..., np.newaxis].astype(np.uint8)
        return eopatch

# ------------------------------- EFFIS burn scar rasterisation ----------------
class BurnScarRasterTask(EOTask):
    def __init__(self, fires_gdf, reference_feature, max_age_days=365):
        self.fires, self.ref, self.max_age = fires_gdf, reference_feature, max_age_days

    def execute(self, eopatch):
        t, h, w = eopatch[self.ref].shape[:3]
        bbox = eopatch.bbox
        transform = from_bounds(bbox.min_x, bbox.min_y, bbox.max_x, bbox.max_y, w, h)
        fires_utm = self.fires.to_crs(bbox.crs.pyproj_crs())
        burn = np.zeros((t, h, w, 1), np.uint8)
        for ti, ts in enumerate(eopatch.timestamp):
            ts64 = np.datetime64(ts)
            sel = fires_utm[fires_utm["fire_date"].values <= ts64]
            if self.max_age is not None and len(sel):
                sel = sel[sel["fire_date"].values >= ts64 - np.timedelta64(self.max_age, "D")]
            if len(sel):
                burn[ti, ..., 0] = rasterize(((g, 1) for g in sel.geometry),
                                             out_shape=(h, w), transform=transform,
                                             fill=0, dtype=np.uint8)
        eopatch[(FeatureType.MASK, "BURN")] = burn
        return eopatch

# --------------------------------------------------------------- final mask --
class CombineMaskTask(EOTask):
    def __init__(self, include_cirrus=False):
        self.include_cirrus = include_cirrus

    def execute(self, eopatch):
        m = eopatch.mask
        cloud = m["OCM_CLOUD_RES"][..., 0].astype(bool)
        if self.include_cirrus:
            cloud |= m["SCL_CIRRUS_RES"][..., 0].astype(bool)
        shadow, water = m["OCM_SHADOW_RES"][..., 0].astype(bool), m["WATER_RES"][..., 0].astype(bool)
        burn, valid = m["BURN"][..., 0].astype(bool), m["dataMask_RES"][..., 0].astype(bool)
        labels = np.full(cloud.shape, MASK_CLEAR, np.uint8)
        labels[water] = MASK_WATER
        labels[burn] = MASK_BURNT
        labels[shadow] = MASK_CLOUD_SHADOW
        labels[cloud] = MASK_CLOUD
        labels[~valid] = MASK_NODATA
        eopatch[(FeatureType.MASK, "LABELS")] = labels[..., np.newaxis]
        return eopatch

# ---------------------------------------------- export: full scenes only ----
def _profile(bbox, h, w, count, dtype, nodata=None):
    return dict(driver="GTiff", height=h, width=w, count=count, dtype=dtype,
                crs=bbox.crs.pyproj_crs(), compress="deflate", nodata=nodata,
                transform=from_bounds(bbox.min_x, bbox.min_y, bbox.max_x, bbox.max_y, w, h))

def _write_bands(path, data_hwc, bbox):
    data = data_hwc[..., EXPORT_IDX]
    h, w, c = data.shape
    if EXPORT_AS_UINT16:
        arr = np.clip(np.round(data * 10000.0), 0, 65535).astype(np.uint16)
        prof = _profile(bbox, h, w, c, "uint16", nodata=0)
    else:
        arr = data.astype(np.float32)
        prof = _profile(bbox, h, w, c, "float32")
    prof.update(predictor=2, tiled=True, blockxsize=512, blockysize=512)
    with rasterio.open(path, "w", **prof) as dst:
        for b in range(c):
            dst.write(arr[..., b], b + 1)
            dst.set_band_description(b + 1, EXPORT_BANDS[b])

def _write_mask(path, mask_hw, bbox):
    h, w = mask_hw.shape
    prof = _profile(bbox, h, w, 1, "uint8", nodata=MASK_NODATA)
    prof.update(tiled=True, blockxsize=512, blockysize=512)
    with rasterio.open(path, "w", **prof) as dst:
        dst.write_colormap(1, MASK_COLORMAP)
        dst.write(mask_hw.astype(np.uint8), 1)

def export_products(eopatch, out_dir, fire_id, resolution, **kwargs):
    written, bbox = [], eopatch.bbox
    bands, labels = eopatch.data["PHISAT2-BANDS"], eopatch.mask["LABELS"]
    for ti, ts in enumerate(eopatch.timestamp):
        ts_str = ts.strftime("%Y-%m-%dT%H-%M-%S")
        scene_dir = os.path.join(out_dir, "acquisitions", fire_id, ts_str)
        os.makedirs(scene_dir, exist_ok=True)
        ip = os.path.join(scene_dir, f"{fire_id}_{ts_str}_phisat2.tif")
        mp = os.path.join(scene_dir, f"{fire_id}_{ts_str}_mask.tif")
        _write_bands(ip, bands[ti], bbox)
        _write_mask(mp, labels[ti, ..., 0], bbox)
        written += [ip, mp]
    return written

# ------------------------------------- SNR/PSF fallback (only if no binary) --
def gaussian_kernel(size=7, sigma=1.0):
    ax = np.arange(size) - size // 2
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))
    return (k / k.sum()).astype(np.float64)

FALLBACK_SNR_VALUES = {"B02": 154, "B03": 168, "B04": 142, "PAN": 180,
                       "B08": 132, "B05": 120, "B06": 127, "B07": 116}
FALLBACK_PSF_KERNELS = {b: gaussian_kernel(7, s) for b, s in zip(
    ["B1","B2","B3","B0","B7","B4","B5","B6"], [0.85,0.9,0.95,0.8,1.15,1.0,1.05,1.1])}
FALLBACK_L_REF = 100.0

def resolve_executable():
    import platform
    system, arch = platform.system().lower(), platform.machine().lower()
    for cand in sorted(glob.glob("executables/*")):
        name = os.path.basename(cand).lower()
        if system.startswith("linux") and "linux" in name: return cand
        if system == "darwin" and ("osx" in name or "darwin" in name):
            if ("arm" in arch) == ("arm" in name): return cand
        if system == "windows" and ("win" in name or name.endswith(".exe")): return cand
    return None

# ---------------------------------------------- Sentinel Hub config & search --
def make_sh_config(cfg):
    sh = SHConfig()
    sh.sh_client_id, sh.sh_client_secret = cfg["sh_client_id"], cfg["sh_client_secret"]
    if cfg["use_cdse"]:
        sh.sh_base_url = CDSE_BASE_URL
        from sentinelhub.download import session as shs
        def _collect_new_token(self):
            return self._fetch_token(shs.DownloadRequest(url=CDSE_TOKEN_URL))
        shs.SentinelHubSession._collect_new_token = _collect_new_token
    return sh

def get_utm_bbox(lat_centre, lon_centre):
    east, north = wgs84_to_utm(lon_centre, lat_centre)
    east, north = 10 * int(east / 10), 10 * int(north / 10)
    return BBox(((east - BBOX_SIZE // 2, north - BBOX_SIZE // 2),
                 (east + BBOX_SIZE // 2, north + BBOX_SIZE // 2)),
                crs=get_utm_crs(lon_centre, lat_centre))

import time

def get_candidate_dates(bbox, fire_date, cfg, sh, max_retries=3, retry_delay=15):
    """Up to cfg['max_candidate_dates'] S2 acquisitions in the post-fire window,
    ascending by scene-wide cloud cover. Retries on transient connection errors."""
    t0 = (fire_date + dt.timedelta(days=cfg["post_fire_min_days"])).date().isoformat()
    t1 = (fire_date + dt.timedelta(days=cfg["post_fire_max_days"])).date().isoformat()
    for attempt in range(1, max_retries + 1):
        try:
            results = list(SentinelHubCatalog(sh).search(
                collection=S2L2A_COLLECTION, bbox=bbox, time=(t0, t1),
                filter=f"eo:cloud_cover < {cfg['maxcc'] * 100:.0f}",
                fields={"include": ["properties.datetime", "properties.eo:cloud_cover"],
                        "exclude": []}))
            results.sort(key=lambda r: r["properties"]["eo:cloud_cover"])
            return [(r["properties"]["datetime"].split("T")[0], r["properties"]["eo:cloud_cover"])
                   for r in results[:cfg["max_candidate_dates"]]]
        except Exception as exc:
            msg = str(exc)
            transient = any(s in msg for s in
                            ("NameResolutionError", "ConnectionError", "Max retries exceeded"))
            if transient and attempt < max_retries:
                log.warning("Catalog search failed (attempt %d/%d, retrying in %ds): %s",
                           attempt, max_retries, retry_delay, msg.splitlines()[0])
                time.sleep(retry_delay)
                continue
            log.warning("Catalog search failed: %s", msg.splitlines()[0])
            return []
    return []

# ------------------------------------------------------- per-fire processing --
def process_fire(fire_row, fires_gdf, cfg, sh):
    """EFFIS fire -> best S2 L1C scene (burn scar not obscured by cloud/shadow;
    clouds/shadows ELSEWHERE in the frame are fine, only overlap with the scar
    is checked) -> official PhiSat-2 sim -> mask -> GeoTIFFs."""
    fire_id = fire_row["fire_id"]
    centroid = fire_row.geometry.centroid
    bbox = get_utm_bbox(centroid.y, centroid.x)

    candidates = get_candidate_dates(bbox, fire_row["fire_date"], cfg, sh)
    if not candidates:
        log.info("[%s] no S2 acquisitions found - skipped", fire_id); return []

    aux = {"processing": {"upsampling": "BICUBIC"}}
    accepted_eop = accepted_date = None
    fallback_eop = fallback_date = None
    fallback_frac = None

    for date, cc in candidates:
        eop = SentinelHubInputTask(
            data_collection=S2L2A_COLLECTION, resolution=S2_RESOLUTION,
            additional_data=[(FeatureType.MASK, "SCL")], maxcc=cfg["maxcc"],
            aux_request_args=aux, config=sh, cache_folder=cfg["cache_folder"],
            time_difference=dt.timedelta(minutes=180),
        )(bbox=bbox, time_interval=(date, date))
        if len(eop.timestamp) == 0:
            continue
        eop = SentinelHubInputTask(
            data_collection=S2L1C_COLLECTION, resolution=S2_RESOLUTION,
            bands_feature=(FeatureType.DATA, "BANDS"),
            additional_data=[(FeatureType.MASK, "dataMask"),
                             (FeatureType.DATA, "sunZenithAngles")],
            bands=S2_BANDS, aux_request_args=aux, config=sh,
            cache_folder=cfg["cache_folder"], time_difference=dt.timedelta(minutes=180),
        )(eopatch=eop)

        eop = WaterMaskTask(cfg["ndwi_threshold"], cfg["use_scl_water"])(eop)
        eop = SCLCloudTask(scl_feature=(FeatureType.MASK, "SCL"))(eop)
        eop = OmniCloudMaskTask(inference_device=cfg["ocm_device"])(eop)
        eop = BurnScarRasterTask(fires_gdf, (FeatureType.DATA, "BANDS"),
                                 max_age_days=cfg["burn_max_age_days"])(eop)

        burn = eop.mask["BURN"][0, ..., 0].astype(bool)
        obscured = (eop.mask["OCM_CLOUD"][0, ..., 0] | eop.mask["OCM_SHADOW"][0, ..., 0]).astype(bool)
        frac = (burn & obscured).sum() / max(burn.sum(), 1)
        log.info("[%s] candidate %s (scene cc=%.0f%%) burn-scar obscured=%.1f%%",
                 fire_id, date, cc, frac * 100)

        if fallback_eop is None or frac < fallback_frac:
            fallback_eop, fallback_date, fallback_frac = eop, date, frac
        if frac <= cfg["burn_cloud_max_fraction"]:
            accepted_eop, accepted_date = eop, date
            break

    if accepted_eop is None:
        if fallback_eop is None:
            log.info("[%s] no usable acquisition - skipped", fire_id); return []
        log.warning("[%s] no candidate met %.0f%% threshold - using best available "
                    "%s (%.1f%% of scar obscured)", fire_id,
                    cfg["burn_cloud_max_fraction"] * 100, fallback_date, fallback_frac * 100)
        accepted_eop, accepted_date = fallback_eop, fallback_date

    eop, date = accepted_eop, accepted_date
    log.info("[%s] fire %s (%.0f ha) -> S2 date %s accepted", fire_id,
             fire_row["fire_date"].date(), fire_row.get("area_ha", np.nan), date)

    level = ProcessingLevels[cfg["processing_level"]]

    eop = (AddMetadataTask(config=sh) if cfg["use_aws_metadata"] else AddMetadataFallbackTask())(eop)
    eop = CalculateRadianceTask((FeatureType.DATA, "BANDS"), (FeatureType.DATA, "BANDS-RAD"))(eop)
    eop = AddPANBandTask((FeatureType.DATA, "BANDS-RAD"), (FeatureType.DATA, "BANDS-RAD-PAN"))(eop)
    eop = MapFeatureTask((FeatureType.MASK, "dataMask"), (FeatureType.MASK, "dataMask"), np.uint8)(eop)

    new_size = (int(BBOX_SIZE / PHISAT2_RESOLUTION),) * 2
    to_resize = {FeatureType.DATA: ["BANDS-RAD-PAN", "sunZenithAngles"],
                 FeatureType.MASK: ["SCL_CLOUD", "SCL_CIRRUS", "SCL_CLOUD_SHADOW",
                                    "dataMask", "OCM_CLOUD", "OCM_SHADOW", "WATER"]}
    for ftype, names in to_resize.items():
        for name in names:
            eop = MapFeatureTask((ftype, name), (ftype, f"{name}_RES"), resize_images,
                                 new_size=new_size, resize_method="nearest")(eop)

    eop = RemoveFeatureTask(
        [(FeatureType.DATA, "BANDS"), (FeatureType.DATA, "BANDS-RAD"),
         (FeatureType.DATA, "BANDS-RAD-PAN"), (FeatureType.DATA, "sunZenithAngles"),
         (FeatureType.MASK, "SCL_CLOUD"), (FeatureType.MASK, "SCL_CLOUD_SHADOW"),
         (FeatureType.MASK, "SCL_CIRRUS"), (FeatureType.MASK, "dataMask"),
         (FeatureType.MASK, "OCM_CLOUD"), (FeatureType.MASK, "OCM_SHADOW"),
         (FeatureType.MASK, "WATER"), (FeatureType.MASK, "BURN")])(eop)

    eop = BandMisalignmentTask((FeatureType.DATA, "BANDS-RAD-PAN_RES"),
                               (FeatureType.DATA, "BANDS-RAD-PAN_SHIFTED"),
                               level, std_sea=6, interpolation_method=cv2.INTER_NEAREST)(eop)
    eop = RemoveFeatureTask([(FeatureType.DATA, "BANDS-RAD-PAN_RES")])(eop)

    eop = CropTask(features_to_crop=[
        (FeatureType.DATA, "BANDS-RAD-PAN_SHIFTED"), (FeatureType.DATA, "sunZenithAngles_RES"),
        (FeatureType.MASK, "SCL_CLOUD_RES"), (FeatureType.MASK, "SCL_CLOUD_SHADOW_RES"),
        (FeatureType.MASK, "SCL_CIRRUS_RES"), (FeatureType.MASK, "dataMask_RES"),
        (FeatureType.MASK, "OCM_CLOUD_RES"), (FeatureType.MASK, "OCM_SHADOW_RES"),
        (FeatureType.MASK, "WATER_RES")])(eop)

    exe = cfg.get("executable_override") or resolve_executable()
    if exe:
        eop = PhisatCalculationTask((FeatureType.DATA, "BANDS-RAD-PAN_SHIFTED"),
                                    (FeatureType.DATA, "L_out_RES"), exe, "SNR").execute(eop)
        eop = PhisatCalculationTask((FeatureType.DATA, "L_out_RES"),
                                    (FeatureType.DATA, "L_out_PSF"), exe, "PSF").execute(eop)
    else:
        log.warning("[%s] no official SNR/PSF binary found - using fallback", fire_id)
        eop = AlternativePhisatCalculationTask(
            input_feature=(FeatureType.DATA, "BANDS-RAD-PAN_SHIFTED"),
            snr_feature=(FeatureType.DATA, "L_out_RES"),
            psf_feature=(FeatureType.DATA, "L_out_PSF"),
            snr_values=FALLBACK_SNR_VALUES, psf_kernel=FALLBACK_PSF_KERNELS,
            l_ref=FALLBACK_L_REF)(eop)

    eop = CalculateReflectanceTask((FeatureType.DATA, "L_out_PSF"),
                                   (FeatureType.DATA, "PHISAT2-BANDS"), level)(eop)
    eop = RemoveFeatureTask([(FeatureType.DATA, "BANDS-RAD-PAN_SHIFTED"),
                             (FeatureType.DATA, "L_out_RES"), (FeatureType.DATA, "L_out_PSF")])(eop)

    eop = BurnScarRasterTask(fires_gdf, (FeatureType.DATA, "PHISAT2-BANDS"),
                             max_age_days=cfg["burn_max_age_days"])(eop)
    eop = CombineMaskTask(include_cirrus=cfg["include_scl_cirrus_as_cloud"])(eop)

    files = export_products(eop, cfg["out_dir"], fire_id, PHISAT2_RESOLUTION)
    log.info("[%s] wrote %d files", fire_id, len(files))
    return files

print("functions defined | export:", EXPORT_BANDS, "| uint16:", EXPORT_AS_UINT16)

functions defined | export: ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08'] | uint16: True


In [ ]:
NEW_EFFIS_PATH = os.path.expanduser(
    "~/projects/iride_onboard-burnscar-mapper/data/PHISAT_5fbf994ef5774ab882bf7239ec8b32f5.zip")

CFG = dict(
    sh_client_id     = os.environ["SH_CLIENT_ID"],
    sh_client_secret = os.environ["SH_CLIENT_SECRET"],
    use_cdse         = True,

    executable_override = os.path.expanduser(
        "~/projects/iride_onboard-burnscar-mapper/executables/phisat2_unix_extracted/phisat2_unix.bin"),

    effis_path   = NEW_EFFIS_PATH,
    countries    = ["IT", "FR", "ES", "EL"],   # EL = Greece (EFFIS/EU code, not GR)
    start_date   = "2020-01-01",
    end_date     = "2026-08-04",
    min_area_ha  = 100.0,

    # -- diversity-set geography --
    southern_france_max_lat = 46.0,   # keep only France fires south of this latitude
                                       # (roughly Nouvelle-Aquitaine/Occitanie/PACA;
                                       # adjust if you want a stricter/looser cut)

    post_fire_min_days = 5, post_fire_max_days = 60, maxcc = 0.70,
    max_candidate_dates = 5, burn_cloud_max_fraction = 0.15,
    processing_level = "L1C", use_aws_metadata = False, ocm_device = None,
    ndwi_threshold = 0.05, use_scl_water = True,
    include_scl_cirrus_as_cloud = False, burn_max_age_days = 365,

    out_dir = "./output", cache_folder = "./temp_data",

    # -- budget: 60% IT, remaining 40% split evenly FR/ES/EL, 50 GB hard cap total --
    total_budget_gb = 50,
    country_quota   = {"IT": 0.60, "FR": 0.40/3, "ES": 0.40/3, "EL": 0.40/3},
    min_free_gb     = 20,
    workers         = 2,
)

from concurrent.futures import ThreadPoolExecutor, as_completed
import copy, gc, itertools, threading

os.makedirs(CFG["out_dir"], exist_ok=True)
sh = make_sh_config(CFG)

# --- load all 4 countries, apply southern-France filter, priority sort ---
fires_all = load_effis(CFG["effis_path"], countries=CFG["countries"],
                       start_date=CFG["start_date"], end_date=CFG["end_date"],
                       min_area_ha=CFG["min_area_ha"])

is_fr = fires_all["country"] == "FR"
southern_ok = ~is_fr | (fires_all.geometry.centroid.y <= CFG["southern_france_max_lat"])
fires = fires_all[southern_ok].reset_index(drop=True)

print("Before southern-France filter:", fires_all["country"].value_counts().to_dict())
print("After southern-France filter: ", fires["country"].value_counts().to_dict())

# IT first, then FR/ES/EL (equal rank -> naturally interleaved by area within
# that tier; per-country budgets below enforce the actual 60/13.3/13.3/13.3 split
# regardless of processing order)
COUNTRY_PRIORITY = {"IT": 0, "FR": 1, "ES": 1, "EL": 1}
fires["_rank"] = fires["country"].map(COUNTRY_PRIORITY).fillna(9)
fires = fires.sort_values(["_rank", "area_ha"], ascending=[True, False]).drop(columns="_rank").reset_index(drop=True)

todo = fires
print(f"{len(todo)} fires queued (post-filter)")
display(todo.head(10))

# --- resume-awareness: measure what's ALREADY on disk, per country ---
fire_id_to_country = dict(zip(fires_all["fire_id"], fires_all["country"]))

def scan_existing_bytes(out_dir, id_to_country):
    written = {c: 0 for c in CFG["country_quota"]}
    unknown = 0
    acq_dir = os.path.join(out_dir, "acquisitions")
    if os.path.isdir(acq_dir):
        for fid in os.listdir(acq_dir):
            fpath = os.path.join(acq_dir, fid)
            if not os.path.isdir(fpath):
                continue
            size = sum(os.path.getsize(os.path.join(r, f))
                      for r, _, fs in os.walk(fpath) for f in fs)
            country = id_to_country.get(fid)
            if country in written:
                written[country] += size
            else:
                unknown += size
    return written, unknown

existing_by_country, unknown_bytes = scan_existing_bytes(CFG["out_dir"], fire_id_to_country)
if unknown_bytes:
    log.warning("%.2f GB on disk belongs to fire_ids not found in the new EFFIS export "
               "(unmatched, not counted against any country quota)", unknown_bytes / 1e9)

country_budget_bytes = {c: frac * CFG["total_budget_gb"] * 1e9
                        for c, frac in CFG["country_quota"].items()}
running_by_country = dict(existing_by_country)  # cumulative total, updated as we go

print("Starting quota state:")
for c in CFG["country_quota"]:
    print(f"  {c}: {running_by_country[c]/1e9:.2f} GB already on disk "
         f"/ {country_budget_bytes[c]/1e9:.2f} GB quota")

def already_done(fire_id):
    d = os.path.join(CFG["out_dir"], "acquisitions", fire_id)
    return os.path.isdir(d) and any(f.endswith("_phisat2.tif")
                                    for _, _, fs in os.walk(d) for f in fs)

def free_gb():
    return shutil.disk_usage(CFG["out_dir"]).free / 1e9

device_cycle = itertools.cycle(["cuda:0", "cuda:1"])
lock = threading.Lock()
stop_flags = {}   # keys: "total", "IT", "FR", "ES", "EL"

def worker(fire_row):
    country, fid = fire_row["country"], fire_row["fire_id"]
    if stop_flags.get("total") or stop_flags.get(country):
        return f"skipped ({country} quota / total budget reached)", 0
    if already_done(fid):
        return "already done", 0
    if free_gb() < CFG["min_free_gb"]:
        stop_flags["total"] = True
        log.error("Free disk below %d GB - stopping new fires", CFG["min_free_gb"])
        return "skipped (low disk)", 0

    with lock:
        if running_by_country[country] >= country_budget_bytes[country]:
            stop_flags[country] = True
            log.warning("%s quota reached (%.2f GB) - stopping new %s fires",
                       country, country_budget_bytes[country] / 1e9, country)
            return f"skipped ({country} quota reached)", 0
        if sum(running_by_country.values()) >= CFG["total_budget_gb"] * 1e9:
            stop_flags["total"] = True
            log.warning("Total %d GB budget reached - stopping", CFG["total_budget_gb"])
            return "skipped (total budget reached)", 0

    cfg = copy.deepcopy(CFG)
    cfg["ocm_device"] = next(device_cycle)
    try:
        files = process_fire(fire_row, fires, cfg, sh)
        size = sum(os.path.getsize(f) for f in files if os.path.exists(f))
        with lock:
            running_by_country[country] += size
        return ("ok" if files else "skipped"), size
    finally:
        gc.collect()
        try:
            import torch; torch.cuda.empty_cache()
        except Exception:
            pass
        shutil.rmtree(CFG["cache_folder"], ignore_errors=True)

index_rows = []
with ThreadPoolExecutor(max_workers=CFG["workers"]) as pool:
    futures = {pool.submit(worker, fire): fire for _, fire in todo.iterrows()}
    for fut in as_completed(futures):
        fire = futures[fut]
        try:
            status, size = fut.result()
        except Exception as exc:
            log.exception("Fire %s failed", fire["fire_id"])
            status, size = f"error: {exc}", 0
        index_rows.append(dict(fire_id=fire["fire_id"], country=fire["country"],
                               fire_date=fire["fire_date"], area_ha=fire["area_ha"],
                               status=status, mb_written=round(size / 1e6, 1)))
        totals = " | ".join(f"{c}:{running_by_country[c]/1e9:.1f}/{country_budget_bytes[c]/1e9:.1f}GB"
                            for c in CFG["country_quota"])
        log.info("progress %d/%d | %s | total %.1f/%d GB | free disk %.0f GB",
                 len(index_rows), len(todo), totals,
                 sum(running_by_country.values()) / 1e9, CFG["total_budget_gb"], free_gb())

idx = pd.DataFrame(index_rows)
idx.to_csv(os.path.join(CFG["out_dir"], "processing_index.csv"), index=False)
todo.to_file(os.path.join(CFG["out_dir"], "fires_index.gpkg"), driver="GPKG")
display(idx)
print(f"Done -> {CFG['out_dir']}")
for c in CFG["country_quota"]:
    print(f"  {c}: {running_by_country[c]/1e9:.2f} GB / {country_budget_bytes[c]/1e9:.2f} GB quota")
print(f"  TOTAL: {sum(running_by_country.values())/1e9:.2f} GB / {CFG['total_budget_gb']} GB budget "
     f"| free disk {free_gb():.0f} GB")

Before southern-France filter: {'IT': 1283, 'ES': 954, 'FR': 478, 'EL': 237}
After southern-France filter:  {'IT': 1283, 'ES': 954, 'FR': 461, 'EL': 237}
2935 fires queued (post-filter)


,geometry,country,fire_date,area_ha,fire_id
0,"MULTIPOLYGON (((8.51001 40.17972, 8.51000 40.1...",IT,2021-07-24 11:00:00,13278,effis_50908
1,"MULTIPOLYGON (((14.21612 37.92935, 14.21606 37...",IT,2021-08-04 11:21:00,9778,effis_52155
2,"MULTIPOLYGON (((15.84076 38.10267, 15.84086 38...",IT,2021-08-04 23:41:00,7096,effis_51974
3,"MULTIPOLYGON (((13.45053 37.66569, 13.45053 37...",IT,2026-07-20 20:14:00,6516,effis_562200
4,"MULTIPOLYGON (((12.75129 38.07266, 12.74944 38...",IT,2025-07-20 11:34:00,5542,effis_275087
5,"MULTIPOLYGON (((14.61116 37.75561, 14.60713 37...",IT,2021-07-02 10:49:00,3762,effis_50209
6,"MULTIPOLYGON (((15.75719 38.00454, 15.75720 38...",IT,2023-07-19 12:25:00,3114,effis_217356
7,"MULTIPOLYGON (((12.74161 38.12921, 12.74200 38...",IT,2020-08-29 20:35:00,3057,effis_42806
8,"MULTIPOLYGON (((14.09535 37.66140, 14.09534 37...",IT,2026-07-21 12:49:00,2999,effis_562296
9,"MULTIPOLYGON (((13.26792 37.93977, 13.26811 37...",IT,2021-07-29 09:40:00,2867,effis_51299


Starting quota state:
  IT: 6.05 GB already on disk / 30.00 GB quota
  FR: 0.00 GB already on disk / 6.67 GB quota
  ES: 0.00 GB already on disk / 6.67 GB quota
  EL: 0.00 GB already on disk / 6.67 GB quota


2026-08-04 16:46:42,862 INFO [effis_50908] candidate 2021-08-21 (scene cc=0%) burn-scar obscured=0.0%
2026-08-04 16:46:42,863 INFO [effis_50908] fire 2021-07-24 (13278 ha) -> S2 date 2021-08-21 accepted
2026-08-04 16:46:43,100 INFO [effis_52155] candidate 2021-09-19 (scene cc=0%) burn-scar obscured=0.0%
2026-08-04 16:46:43,101 INFO [effis_52155] fire 2021-08-04 (9778 ha) -> S2 date 2021-09-19 accepted
2026-08-04 16:47:26,925 INFO [effis_50908] wrote 2 files
2026-08-04 16:47:32,305 INFO progress 1/2935 | IT:6.2/30.0GB | FR:0.0/6.7GB | ES:0.0/6.7GB | EL:0.0/6.7GB | total 6.2/50 GB | free disk 168 GB
2026-08-04 16:47:32,329 INFO [effis_52155] wrote 2 files
2026-08-04 16:47:32,606 INFO progress 2/2935 | IT:6.3/30.0GB | FR:0.0/6.7GB | ES:0.0/6.7GB | EL:0.0/6.7GB | total 6.3/50 GB | free disk 168 GB
2026-08-04 16:47:32,608 INFO progress 3/2935 | IT:6.3/30.0GB | FR:0.0/6.7GB | ES:0.0/6.7GB | EL:0.0/6.7GB | total 6.3/50 GB | free disk 168 GB
2026-08-04 16:47:32,608 INFO progress 4/2935 | IT:6.